In [ ]:
# Import required packages
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.base import BaseEstimator
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from typing import Callable


## Import Data

In [2]:
# import data 
url = "https://github.com/bozercavdar/kickstarter-project/releases/download/v1.0/kickstarter_data_OSF.csv"
local_path = "dataset.csv"

# UNCOMMENT THE LINE BELOW IF YOU DON'T HAVE THE DATASET IN THE LOCAL DIRECTORY
# df = pd.read_csv(url)
df = pd.read_csv(local_path)

In [3]:
# Filter only those required columns
selected_df = df[['uid', 'blurb', 'goal', 'state', 'usd_pledged', 'category']]
# Add column that states if a project is succesful or not
selected_df['success'] = selected_df['state'] == 'successful'
selected_df.head()

/tmp/ipykernel_21485/2469478530.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df['success'] = selected_df['state'] == 'successful'


,uid,blurb,goal,state,usd_pledged,category,success
0,1,This project is designed to help protect the e...,2500.0,failed,0.0,Music,False
1,2,Help us built a sustainable studio & eliminate...,25000.0,failed,1.0,Technology,False
2,3,"""If I paint something, I don't want to have to...",5000.0,failed,5.0,Art,False
3,4,Our free app will allow you pool reservations ...,12000.0,failed,0.0,Food,False
4,5,Prohibition themed Gastro Pub and After Dark S...,20000.0,failed,0.0,Food,False


## Analysis

In [4]:
# Load tokenizer and model
model_name = "siebert/sentiment-roberta-large-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [5]:
sample_df = selected_df.sample(10000, random_state=41)
sample_df

,uid,blurb,goal,state,usd_pledged,category,success
155006,189791,Spend less time searching your purse and more ...,20000.0,successful,23088.608880,Fashion,True
69020,77192,Kevin Jenkins will choreograph and direct a ne...,3000.0,successful,3635.318822,web,True
75028,83929,Finalist in the Telio National Design Competit...,800.0,successful,1126.748389,Fashion,True
19323,21429,What are the real stories behind these unique ...,5000.0,failed,450.000000,Journalism,False
22301,24722,"High cost-performance, Fastest, Highest accura...",50000.0,failed,37382.000000,Technology,False
...,...,...,...,...,...,...,...
33322,37060,"This book is for, “When Life gets Crappy!” Lik...",500.0,failed,1.000000,Publishing,False
21788,24154,A book of essays on Africa and urban poverty.,575000.0,failed,5.102393,Journalism,False
52221,58290,A poster campaign for the American moment.,86.0,failed,0.000000,Design,False
92761,104716,Telling the story of the city through remarkab...,50000.0,successful,49636.098520,Photography,True


In [6]:
# Function that extracts textual feature with the given function and uses them to train classifiers
def classifier(df: pd.DataFrame, feature_extractor: Callable, cols: list, 
               estimators: list[BaseEstimator]):
    df = df.join(df["blurb"].apply(feature_extractor))
    
    X = df[cols].copy()
    y = df["success"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=41, stratify=y
    )

    total_results = None

    for estimator in estimators:
        estimator.fit(X_train, y_train)

        y_pred = estimator.predict(X_test)
        # y_pred_prob = estimator.predict_proba(X_test)[:, 1]

        results = pd.DataFrame({
            "Metric": ["success_pred_rate", "accuracy", "precision", "recall"],
            f"{estimator.__class__.__name__}": [
                y_pred.mean(),
                accuracy_score(y_test, y_pred),
                precision_score(y_test, y_pred),
                recall_score(y_test, y_pred)
                # roc_auc_score(y_test, y_pred_prob)
            ]
        })

        if total_results is None:
            total_results = results
        else:
            total_results = pd.merge(total_results, results, on="Metric")

    return total_results


In [10]:
def siebert_sentiment(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True)
    with torch.no_grad():
        output = model(**tokens)

    logits = output.logits
    probs = torch.softmax(logits, dim=1).numpy()[0]

    return pd.Series({
        "sieber_pos_prob": probs[1],     # Probability of positive sentiment
        "sieber_neg_prob": probs[0],     # Probability of negative sentiment
        "sieber_label": "positive" if probs[1] > probs[0] else "negative"
    })

col_names = ["sieber_pos_prob", "sieber_neg_prob"]

classifier(sample_df, siebert_sentiment, col_names, estimators=[LogisticRegression(), RandomForestClassifier(), KNeighborsClassifier(), SVC()])

,Metric,LogisticRegression,RandomForestClassifier,KNeighborsClassifier,SVC
0,success_pred_rate,1.000000,0.581000,0.654333,1.000000
1,accuracy,0.583667,0.510000,0.516667,0.583667
2,precision,0.583667,0.580608,0.576668,0.583667
3,recall,1.000000,0.577955,0.646488,1.000000
